# Causal intervention diagnostic

This notebook compares hidden-state answer injection with textual premise injection. Both diagnostics use the known edit that owns each two-hop causal question, so retrieval quality is held out of the intervention test. No oracle behavior is used by benchmark generation.

The hidden-state diagnostic compares base output, normal Mnemo routing, and forced oracle routing. The contextual diagnostics compare unmodified prompts with the same simple and causal prompts preceded by either their correct edit fact or an unrelated fact. `run_natural_context_experiment()` uses ordinary `Given ...` prose without `New Fact`, `Prompt`, or `Answer` labels. Results are written under `data/causality/intervention/`, separate from evaluation artifacts.

In [ ]:
import hashlib
import json
import random
import re
import sys
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.editors.mnemo import Mnemo
from src.editors.editor_utils import EditRequest
from src.utils.devices import get_model_device

In [ ]:
def load_records(path: Path) -> list[dict]:
    records = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row.get('edit'), dict) or not isinstance(row.get('causality'), list):
                raise ValueError(f'Invalid causality row at line {line_number}')
            records.append(row)
    return records

def owner_key(edit: dict) -> tuple[str, str]:
    return (str(edit['subject']['id']), str(edit['relation']['id']))

def dependency_keys(query: dict) -> list[tuple[str, str]]:
    return [(str(item['subject_id']), str(item['relation_id'])) for item in query.get('dependency_edit_keys', [])]

def collect_queries(records: list[dict]) -> tuple[dict[tuple[str, str], int], list[dict]]:
    owners = {owner_key(row['edit']): index for index, row in enumerate(records)}
    queries = []
    for row_index, row in enumerate(records):
        for query_index, query in enumerate(row['causality']):
            keys = dependency_keys(query)
            # The default data policy is independent edits: one causal query has one owner.
            if len(keys) != 1 or keys[0] not in owners:
                continue
            queries.append({
                'row_index': row_index,
                'query_index': query_index,
                'query': query,
                'owner_key': keys[0],
                'owner_row_index': owners[keys[0]],
            })
    return owners, queries

In [ ]:
def stratified_sample(items: list[dict], per_stratum: int, seed: int) -> list[dict]:
    groups = defaultdict(list)
    for item in items:
        query = item['query']
        groups[(query['source_edit_position'], query['origin'])].append(item)
    rng = random.Random(seed)
    selected = []
    for group_key in sorted(groups):
        group = list(groups[group_key])
        rng.shuffle(group)
        selected.extend(group[:per_stratum])
    selected.sort(key=lambda item: (item['row_index'], item['query_index']))
    return selected

def prepare_experiment() -> None:
    global records, owners, queries, selected_queries
    global required_owner_keys, bank_keys, bank_rows, bank_index, context_items
    records = load_records(DATA_PATH)
    owners, queries = collect_queries(records)
    selected_queries = stratified_sample(queries, QUERY_LIMIT_PER_STRATUM, SEED)
    required_owner_keys = sorted({item['owner_key'] for item in selected_queries})
    if len(required_owner_keys) > MAX_EDIT_BANK:
        raise ValueError(
            f'Selected queries require {len(required_owner_keys)} owners, exceeding MAX_EDIT_BANK={MAX_EDIT_BANK}. '
            'Lower QUERY_LIMIT_PER_STRATUM or increase the explicit bank cap.'
        )
    distractor_keys = sorted(set(owners) - set(required_owner_keys))
    random.Random(SEED + 1).shuffle(distractor_keys)
    bank_keys = required_owner_keys + distractor_keys[:MAX_EDIT_BANK - len(required_owner_keys)]
    random.Random(SEED + 2).shuffle(bank_keys)
    bank_rows = [records[owners[key]] for key in bank_keys]
    bank_index = {key: index for index, key in enumerate(bank_keys)}
    for item in selected_queries:
        item['owner_memory_index'] = bank_index[item['owner_key']]
    context_items = build_context_items()
    print(f'loaded edits={len(records)}, eligible queries={len(queries)}')
    print(Counter((item['query']['source_edit_position'], item['query']['origin']) for item in queries))
    print(f'selected queries={len(selected_queries)}, edit bank={len(bank_rows)}')
    print(Counter((item['query']['source_edit_position'], item['query']['origin']) for item in selected_queries))
    print(f'context queries={len(context_items)}')
    print(Counter(item['query_type'] for item in context_items))

In [ ]:
def build_mnemo() -> Mnemo:
    config = load_config(MNEMO_CONFIG_PATH)
    config.external_only = EXTERNAL_ONLY
    return Mnemo(config)

def clean_text(text: str) -> str:
    text = text.strip().casefold()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

def generate_texts(
    editor: Mnemo,
    prompts: list[str],
    *,
    forced_indices: list[int] | None = None,
    strength: float | None = None,
) -> list[str]:
    if not prompts:
        return []
    if forced_indices is not None and len(forced_indices) != len(prompts):
        raise ValueError('Forced indices must align with prompts')
    tokenizer = editor.tokenizer
    model = editor.model
    original_padding = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    predictions = []
    try:
        for start in range(0, len(prompts), BATCH_SIZE):
            batch = prompts[start:start + BATCH_SIZE]
            batch_indices = None if forced_indices is None else forced_indices[start:start + BATCH_SIZE]
            inputs = tokenizer(batch, return_tensors='pt', padding=True).to(editor.device)
            generation_inputs = {
                **inputs,
                'max_new_tokens': MAX_NEW_TOKENS,
                'repetition_penalty': REPETITION_PENALTY,
                'do_sample': False,
                'pad_token_id': tokenizer.pad_token_id,
                'eos_token_id': tokenizer.eos_token_id,
                'routing_prompts': batch,
            }
            patch_context = (
                nullcontext()
                if batch_indices is None
                else editor.force_patch(batch_indices, strength=strength)
            )
            with patch_context, torch.inference_mode():
                output_ids = model.generate(**generation_inputs)
            prompt_length = inputs['input_ids'].shape[1]
            predictions.extend(tokenizer.batch_decode(output_ids[:, prompt_length:], skip_special_tokens=True))
    finally:
        tokenizer.padding_side = original_padding
    return [prediction.strip() for prediction in predictions]

def answer_match(prediction: str, answers: list[str]) -> tuple[bool, str | None]:
    value = clean_text(prediction)
    if not value:
        return False, None
    normalized = [clean_text(answer) for answer in answers if clean_text(answer)]
    for answer in normalized:
        if value == answer or value.startswith(answer + ' '):
            return True, 'prefix'
    for answer in normalized:
        if f' {answer} ' in f' {value} ':
            return True, 'containment'
    return False, None

In [ ]:
def apply_owner_edits(editor: Mnemo) -> None:
    for row in tqdm(bank_rows, desc='Applying owner edits'):
        edit = row['edit']
        editor.edit(EditRequest(
            subject=edit['subject']['str'],
            prompt=edit['prompt'],
            answer=edit['answers'][0],
        ))
    if editor.external_memory_size != len(bank_rows):
        raise RuntimeError('The Mnemo memory bank is not aligned with sampled owners')

def run_predictions(editor: Mnemo) -> tuple[list[str], list[str], dict[str, list[str]]]:
    query_prompts = [item['query']['prompt'] for item in selected_queries]
    base_predictions = generate_texts(editor, query_prompts)
    apply_owner_edits(editor)
    normal_predictions = generate_texts(editor, query_prompts)
    owner_indices = [item['owner_memory_index'] for item in selected_queries]
    oracle_predictions = {
        str(strength): generate_texts(
            editor,
            query_prompts,
            forced_indices=owner_indices,
            strength=strength,
        )
        for strength in ORACLE_STRENGTHS
    }
    return base_predictions, normal_predictions, oracle_predictions

In [ ]:
# Kept as a separate definition cell so the expensive run is explicit below.
def compare_scores(base_scores: list[dict], candidate_scores: list[dict]) -> dict:
    groups = defaultdict(lambda: {'count': 0, 'incorrect_to_correct': 0, 'correct_to_incorrect': 0, 'net_gain': 0})
    for item, base, candidate in zip(selected_queries, base_scores, candidate_scores, strict=True):
        query = item['query']
        keys = ('overall', f'position_{query["source_edit_position"]}')
        for key in keys:
            groups[key]['count'] += 1
            if not base['matched'] and candidate['matched']:
                groups[key]['incorrect_to_correct'] += 1
            if base['matched'] and not candidate['matched']:
                groups[key]['correct_to_incorrect'] += 1
    for values in groups.values():
        values['net_gain'] = values['incorrect_to_correct'] - values['correct_to_incorrect']
    return dict(sorted(groups.items()))

In [ ]:
def score_selected_predictions(
    base_predictions: list[str],
    normal_predictions: list[str],
    oracle_predictions: dict[str, list[str]],
) -> tuple[list[dict], list[dict], dict[str, list[dict]]]:
    base_scores = score_condition(base_predictions)
    normal_scores = score_condition(normal_predictions)
    oracle_scores = {strength: score_condition(predictions) for strength, predictions in oracle_predictions.items()}
    return base_scores, normal_scores, oracle_scores

In [ ]:
def score_condition(predictions: list[str]) -> list[dict]:
    scored = []
    for item, prediction in zip(selected_queries, predictions, strict=True):
        matched, mode = answer_match(prediction, item['query']['answers'])
        scored.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
    return scored

def summarize(scores: list[dict]) -> dict:
    groups = defaultdict(list)
    for item, score in zip(selected_queries, scores, strict=True):
        query = item['query']
        groups['overall'].append(score['matched'])
        groups[f'position_{query["source_edit_position"]}'].append(score['matched'])
        groups[f'{query["origin"]}:position_{query["source_edit_position"]}'].append(score['matched'])
    return {key: {'count': len(values), 'accuracy': sum(values) / len(values) if values else None} for key, values in sorted(groups.items())}

In [ ]:
def build_context_items() -> list[dict]:
    if len(bank_rows) < 2:
        raise ValueError('Context intervention requires at least two bank rows')
    items = []
    for sample_index, selected in enumerate(selected_queries):
        edit = records[selected['owner_row_index']]['edit']
        common = {
            'sample_index': sample_index,
            'owner_key': selected['owner_key'],
            'owner_memory_index': selected['owner_memory_index'],
            'wrong_memory_index': (selected['owner_memory_index'] + 1) % len(bank_rows),
            'edit': edit,
        }
        items.append({
            **common,
            'query_type': 'simple',
            'prompt': edit['prompt'],
            'answers': edit['answers'],
            'source_edit_position': None,
            'origin': None,
        })
        query = selected['query']
        items.append({
            **common,
            'query_type': 'causal',
            'prompt': query['prompt'],
            'answers': query['answers'],
            'source_edit_position': query['source_edit_position'],
            'origin': query['origin'],
        })
    return items

def render_context(edit: dict, query: str, template: str) -> str:
    edit_prompt = edit['prompt'].strip()
    edit_answer = edit['answers'][0].strip()
    if template == 'new_fact':
        return f'New Fact: {edit_prompt} {edit_answer}\nPrompt: {query}'
    if template == 'qa':
        return (
            f'Updated knowledge:\nPrompt: {edit_prompt}\nAnswer: {edit_answer}'
            f'\n\nPrompt: {query}\nAnswer:'
        )
    if template == 'natural':
        if edit_prompt.endswith('?'):
            premise = f'that the answer to "{edit_prompt}" is "{edit_answer}"'
        else:
            premise = f'the new fact that {edit_prompt} {edit_answer}'
        return f'Given {premise}, {query}'
    raise ValueError(f'Unsupported context template: {template!r}')

def build_context_model():
    config = load_config(MNEMO_CONFIG_PATH)
    model_kwargs = {'dtype': torch.bfloat16 if config.fp16 else torch.float32}
    if config.model_parallel:
        model_kwargs['device_map'] = 'auto'
    model = AutoModelForCausalLM.from_pretrained(config.model_name, **model_kwargs)
    if not config.model_parallel:
        model = model.to(config.device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
    device = get_model_device(model, fallback=config.device)
    return model, tokenizer, device

In [ ]:
def generate_context_texts(model, tokenizer, device, prompts: list[str]) -> list[str]:
    original_padding = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    predictions = []
    try:
        for start in range(0, len(prompts), BATCH_SIZE):
            batch = prompts[start:start + BATCH_SIZE]
            inputs = tokenizer(batch, return_tensors='pt', padding=True).to(device)
            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    repetition_penalty=REPETITION_PENALTY,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            prompt_length = inputs['input_ids'].shape[1]
            predictions.extend(
                tokenizer.batch_decode(output_ids[:, prompt_length:], skip_special_tokens=True)
            )
    finally:
        tokenizer.padding_side = original_padding
    return [prediction.strip() for prediction in predictions]

def build_context_prompts(condition: str) -> list[str]:
    if condition == 'base':
        return [item['prompt'] for item in context_items]
    context_kind, template = condition.split(':', maxsplit=1)
    prompts = []
    for item in context_items:
        if context_kind == 'correct':
            edit = item['edit']
        elif context_kind == 'wrong':
            edit = bank_rows[item['wrong_memory_index']]['edit']
        else:
            raise ValueError(f'Unsupported context condition: {condition!r}')
        prompts.append(render_context(edit, item['prompt'], template))
    return prompts

def context_conditions() -> list[str]:
    conditions = ['base']
    conditions.extend(f'correct:{template}' for template in CONTEXT_TEMPLATES)
    if INCLUDE_WRONG_CONTEXT:
        conditions.extend(f'wrong:{template}' for template in CONTEXT_TEMPLATES)
    return conditions

In [ ]:
def score_context_condition(predictions: list[str]) -> list[dict]:
    scores = []
    for item, prediction in zip(context_items, predictions, strict=True):
        matched, mode = answer_match(prediction, item['answers'])
        scores.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
    return scores

def context_group_keys(item: dict) -> tuple[str, ...]:
    keys = ['overall', item['query_type']]
    if item['query_type'] == 'causal':
        keys.append(f'causal:position_{item["source_edit_position"]}')
        keys.append(f'{item["origin"]}:position_{item["source_edit_position"]}')
    return tuple(keys)

def summarize_context(scores: list[dict]) -> dict:
    groups = defaultdict(list)
    for item, score in zip(context_items, scores, strict=True):
        for key in context_group_keys(item):
            groups[key].append(score['matched'])
    return {
        key: {'count': len(values), 'accuracy': sum(values) / len(values) if values else None}
        for key, values in sorted(groups.items())
    }

def compare_context_scores(base_scores: list[dict], candidate_scores: list[dict]) -> dict:
    groups = defaultdict(lambda: {
        'count': 0,
        'incorrect_to_correct': 0,
        'correct_to_incorrect': 0,
        'net_gain': 0,
    })
    for item, base, candidate in zip(context_items, base_scores, candidate_scores, strict=True):
        for key in context_group_keys(item):
            groups[key]['count'] += 1
            if not base['matched'] and candidate['matched']:
                groups[key]['incorrect_to_correct'] += 1
            if base['matched'] and not candidate['matched']:
                groups[key]['correct_to_incorrect'] += 1
    for values in groups.values():
        values['net_gain'] = values['incorrect_to_correct'] - values['correct_to_incorrect']
    return dict(sorted(groups.items()))

In [ ]:
def context_fingerprint() -> str:
    payload = '\n'.join(
        f'{item["query_type"]}\t{item["prompt"]}' for item in context_items
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def build_context_payload(
    prompts: dict[str, list[str]],
    predictions: dict[str, list[str]],
    scores: dict[str, list[dict]],
) -> dict:
    rows = []
    for index, item in enumerate(context_items):
        rows.append({
            'sample_index': item['sample_index'],
            'query_type': item['query_type'],
            'prompt': item['prompt'],
            'answers': item['answers'],
            'source_edit_position': item['source_edit_position'],
            'origin': item['origin'],
            'owner_key': list(item['owner_key']),
            'owner_memory_index': item['owner_memory_index'],
            'wrong_memory_index': item['wrong_memory_index'],
            'edit': {
                'subject': item['edit']['subject']['str'],
                'prompt': item['edit']['prompt'],
                'answer': item['edit']['answers'][0],
            },
            'conditions': {
                condition: {
                    'input': prompts[condition][index],
                    **scores[condition][index],
                }
                for condition in prompts
            },
        })
    base_scores = scores['base']
    return {
        'schema_version': 1,
        'experiment': 'contextual_premise_oracle',
        'model_key': MODEL_KEY,
        'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
        'seed': SEED,
        'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
        'edit_bank_size': len(bank_rows),
        'context_templates': list(CONTEXT_TEMPLATES),
        'include_wrong_context': INCLUDE_WRONG_CONTEXT,
        'dataset_fingerprint': context_fingerprint(),
        'summary': {condition: summarize_context(value) for condition, value in scores.items()},
        'comparisons_vs_base': {
            condition: compare_context_scores(base_scores, value)
            for condition, value in scores.items()
            if condition != 'base'
        },
        'rows': rows,
    }

def run_context_experiment() -> None:
    prepare_experiment()
    model, tokenizer, device = build_context_model()
    try:
        prompts = {condition: build_context_prompts(condition) for condition in context_conditions()}
        predictions = {}
        for condition, condition_prompts in prompts.items():
            print(f'generating condition={condition} records={len(condition_prompts)}')
            predictions[condition] = generate_context_texts(
                model, tokenizer, device, condition_prompts
            )
        scores = {
            condition: score_context_condition(value)
            for condition, value in predictions.items()
        }
        payload = build_context_payload(prompts, predictions, scores)
        atomic_save(payload, CONTEXT_OUTPUT_PATH)
        for condition, summary in payload['summary'].items():
            print(condition, summary)
        print(f'saved {CONTEXT_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def run_natural_context_experiment() -> None:
    conditions = ('base', 'correct:natural', 'wrong:natural')
    prepare_experiment()
    model, tokenizer, device = build_context_model()
    try:
        prompts = {condition: build_context_prompts(condition) for condition in conditions}
        predictions = {}
        for condition, condition_prompts in prompts.items():
            print(f'generating condition={condition} records={len(condition_prompts)}')
            predictions[condition] = generate_context_texts(
                model, tokenizer, device, condition_prompts
            )
        scores = {
            condition: score_context_condition(value)
            for condition, value in predictions.items()
        }
        payload = build_context_payload(prompts, predictions, scores)
        payload['experiment'] = 'natural_contextual_premise_oracle'
        payload['context_templates'] = ['natural']
        atomic_save(payload, NATURAL_OUTPUT_PATH)
        for condition, summary in payload['summary'].items():
            print(condition, summary)
        print(f'saved {NATURAL_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def build_prerequisite_items() -> list[dict]:
    items = []
    for sample_index, selected in enumerate(selected_queries):
        query = selected['query']
        hops = query.get('hops')
        if not isinstance(hops, list) or len(hops) != 2:
            raise ValueError(f'Expected exactly two hops for sample {sample_index}')
        source_position = query['source_edit_position']
        if source_position not in (0, 1):
            raise ValueError(f'Invalid source edit position for sample {sample_index}: {source_position!r}')
        if not hops[source_position].get('is_source_edit'):
            raise ValueError(f'Source hop is not marked as the edit for sample {sample_index}')
        prerequisite_index = 1 - source_position
        prerequisite = hops[prerequisite_index]
        if prerequisite.get('is_source_edit'):
            raise ValueError(f'Prerequisite hop is marked as the source edit for sample {sample_index}')
        items.append({
            'sample_index': sample_index,
            'causal_prompt': query['prompt'],
            'source_edit_position': source_position,
            'origin': query['origin'],
            'owner_key': selected['owner_key'],
            'prerequisite_hop_index': prerequisite_index,
            'prompt': prerequisite['prompt'],
            'answers': prerequisite['answers'],
        })
    return items

def summarize_prerequisites(items: list[dict], scores: list[dict]) -> dict:
    groups = defaultdict(list)
    for item, score in zip(items, scores, strict=True):
        position = item['source_edit_position']
        groups['overall'].append(score['matched'])
        groups[f'position_{position}'].append(score['matched'])
        groups[f'{item["origin"]}:position_{position}'].append(score['matched'])
    return {
        key: {'count': len(values), 'accuracy': sum(values) / len(values) if values else None}
        for key, values in sorted(groups.items())
    }

def run_prerequisite_experiment() -> None:
    prepare_experiment()
    prerequisite_items = build_prerequisite_items()
    model, tokenizer, device = build_context_model()
    try:
        prompts = [item['prompt'] for item in prerequisite_items]
        print(f'generating prerequisite records={len(prompts)}')
        predictions = generate_context_texts(model, tokenizer, device, prompts)
        scores = []
        for item, prediction in zip(prerequisite_items, predictions, strict=True):
            matched, mode = answer_match(prediction, item['answers'])
            scores.append({'matched': matched, 'match_mode': mode, 'prediction': prediction})
        payload = {
            'schema_version': 1,
            'experiment': 'prerequisite_hop_control',
            'model_key': MODEL_KEY,
            'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
            'seed': SEED,
            'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
            'causal_dataset_fingerprint': fingerprint(selected_queries),
            'summary': summarize_prerequisites(prerequisite_items, scores),
            'rows': [
                {
                    **item,
                    'owner_key': list(item['owner_key']),
                    **score,
                }
                for item, score in zip(prerequisite_items, scores, strict=True)
            ],
        }
        atomic_save(payload, PREREQUISITE_OUTPUT_PATH)
        print(payload['summary'])
        print(f'saved {PREREQUISITE_OUTPUT_PATH}')
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
def fingerprint(items: list[dict]) -> str:
    payload = '\n'.join(item['query']['prompt'] for item in items).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def atomic_save(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=True, indent=2), encoding='utf-8')
    temporary.replace(path)

def build_payload(base_predictions, normal_predictions, oracle_predictions, base_scores, normal_scores, oracle_scores) -> dict:
    result_rows = []
    for index, item in enumerate(selected_queries):
        query = item['query']
        result_rows.append({
            'query_index': index,
            'prompt': query['prompt'],
            'answers': query['answers'],
            'source_edit_position': query['source_edit_position'],
            'origin': query['origin'],
            'owner_key': list(item['owner_key']),
            'owner_memory_index': item['owner_memory_index'],
            'base_prediction': base_predictions[index],
            'normal_prediction': normal_predictions[index],
            'base_match': base_scores[index],
            'normal_match': normal_scores[index],
            'oracle': {
                strength: {**oracle_scores[strength][index], 'prediction': oracle_predictions[strength][index]}
                for strength in oracle_predictions
            },
        })
    return {
        'schema_version': 1,
        'experiment': 'causal_intervention_oracle',
        'model_key': MODEL_KEY,
        'data_path': str(DATA_PATH.relative_to(PROJECT_ROOT)),
        'seed': SEED,
        'query_limit_per_stratum': QUERY_LIMIT_PER_STRATUM,
        'edit_bank_size': len(bank_rows),
        'oracle_strengths': list(ORACLE_STRENGTHS),
        'dataset_fingerprint': fingerprint(selected_queries),
        'summary': {
            'base': summarize(base_scores),
            'normal': summarize(normal_scores),
            'oracle': {strength: summarize(scores) for strength, scores in oracle_scores.items()},
        },
        'comparisons': {
            'normal_vs_base': compare_scores(base_scores, normal_scores),
            'oracle_vs_base': {strength: compare_scores(base_scores, scores) for strength, scores in oracle_scores.items()},
        },
        'edit_bank': [
            {
                'memory_index': index,
                'owner_key': list(key),
                'subject': row['edit']['subject']['str'],
                'prompt': row['edit']['prompt'],
                'answer': row['edit']['answers'][0],
            }
            for index, (key, row) in enumerate(zip(bank_keys, bank_rows, strict=True))
        ],
        'rows': result_rows,
    }

def run_oracle_experiment() -> None:
    prepare_experiment()
    editor = build_mnemo()
    try:
        base_predictions, normal_predictions, oracle_predictions = run_predictions(editor)
        base_scores, normal_scores, oracle_scores = score_selected_predictions(
            base_predictions, normal_predictions, oracle_predictions
        )
        payload = build_payload(
            base_predictions, normal_predictions, oracle_predictions,
            base_scores, normal_scores, oracle_scores,
        )
        atomic_save(payload, ORACLE_OUTPUT_PATH)
        print('base', payload['summary']['base'])
        print('normal', payload['summary']['normal'])
        for strength, summary in payload['summary']['oracle'].items():
            print(f'oracle strength={strength}', summary)
        print(f'saved {ORACLE_OUTPUT_PATH}')
    finally:
        editor.reset()

Interpretation: the hidden-state result separates routing from vector transfer. The contextual result tests whether one fact-prefix mechanism can handle both direct edit prompts and causal questions. Correct context should improve both query types, while unrelated context should remain close to base behavior. Position 0 and position 1 remain separate because second-hop edit ownership may be hidden from the causal question.

In [ ]:
# Experiment controls. Increase limits only after the smoke run looks sensible.
MODEL_KEY = 'qwen2.5-7b'
MNEMO_CONFIG_PATH = PROJECT_ROOT / 'src/config/mnemo/qwen2.5-7b.yaml'
DATA_PATH = PROJECT_ROOT / 'data/datasets/uniedit/clean/causality.jsonl'
ORACLE_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'oracle.json'
CONTEXT_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'context.json'
NATURAL_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'natural.json'
PREREQUISITE_OUTPUT_PATH = PROJECT_ROOT / 'data/causality/intervention' / MODEL_KEY / 'prerequisite.24.json'
SEED = 0
QUERY_LIMIT_PER_STRATUM = 24
MAX_EDIT_BANK = 100
ORACLE_STRENGTHS = (0.25, 0.5, 0.75, 1.0)
BATCH_SIZE = 4
MAX_NEW_TOKENS = 24
REPETITION_PENALTY = 1.5
CONTEXT_TEMPLATES = ('new_fact', 'qa')
INCLUDE_WRONG_CONTEXT = True

# The hidden-state diagnostic deliberately starts in external-only mode.
EXTERNAL_ONLY = True

print(f'data={DATA_PATH}')
print(f'oracle_output={ORACLE_OUTPUT_PATH}')
print(f'context_output={CONTEXT_OUTPUT_PATH}')
print(f'natural_output={NATURAL_OUTPUT_PATH}')
print(f'prerequisite_output={PREREQUISITE_OUTPUT_PATH}')
print(f'query_limit_per_stratum={QUERY_LIMIT_PER_STRATUM}, bank_cap={MAX_EDIT_BANK}')

In [ ]:
# Uncomment the action you want to run.
# run_oracle_experiment()
# run_context_experiment()
# run_natural_context_experiment()
# run_prerequisite_experiment()